In [1]:
# --- 1. Core Data Manipulation & System Utilities ---
import atexit
import tempfile
import pandas as pd
import numpy as np

# --- 2. Pipeline, Caching & Preprocessing Utilities ---
from joblib import Memory
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
)

# --- 3. Model Selection & Validation ---
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    TunedThresholdClassifierCV,
    train_test_split,
)

# --- 4. Classification Estimators ---
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

# --- 5. Evaluation Metrics & Warning Suppression ---
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)

import warnings; warnings.filterwarnings("ignore")

## Read Data CSV

In [2]:
# --- 1. Load Training Dataset ---
sessions = pd.read_csv("Raw_Data/challenge2_train.csv")

# --- 2. Inspect Schema & Feature Data Types ---
display(sessions.head())
sessions.info()

# --- 3. Inspect Target Class Distribution & Imbalance Ratio ---
print()
display(sessions["Revenue"].value_counts(dropna=False))
print()
display(sessions["Revenue"].value_counts(normalize=True, dropna=False))

,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


<class 'pandas.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   str    
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                   9247 non-n

Revenue
False    7816
True     1431
Name: count, dtype: int64

Revenue
False    0.845247
True     0.154753
Name: proportion, dtype: float64

## Train/Validation Splits 

In [3]:
# --- 1. Separate Predictors and Target (Drop Identifier Column) ---
X = sessions.drop(columns=["Revenue", "ID"])
y = sessions["Revenue"].copy()

# --- 2. Perform Stratified 80/20 Train/Validation Split ---
# Preserves the ~84.5% False / ~15.5% True target distribution across both splits
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# --- 3. Verify Class Proportions Across Development and Holdout Splits ---
display(pd.DataFrame({
    "development": y_train.value_counts(normalize=True),
    "holdout": y_valid.value_counts(normalize=True),
}))

,development,holdout
Revenue,,
False,0.845208,0.845405
True,0.154792,0.154595


## Model Pipeline

In [4]:
# --- 1. Partition Categorical and Numerical Feature Columns ---
categorical = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

numerical = [column for column in X_train.columns if column not in categorical]

# --- 2. Build Leakage-Free Preprocessing Pipeline ---
# Applies Yeo-Johnson power scaling to skewed numerical features and groups rare (<1%)
# categorical levels; imputers are omitted since all 9,247 rows have zero missing values
preprocessor = ColumnTransformer(
    transformers=[
        ("num", PowerTransformer(method="yeo-johnson", standardize=True), numerical),
        ("cat", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=0.01,
            sparse_output=False,
        ), categorical),
    ],
    remainder="drop",
)

# --- 3. Configure Shared Cross-Validation Splitter & Positive-Class F1 Scorer ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 4. Configure Auto-Cleaning Temporary Pipeline Cache ---
# Caches fitted fold transformers in an OS temp folder that auto-deletes on kernel exit
cache_dir = tempfile.TemporaryDirectory(prefix="sklearn_cache_")
atexit.register(cache_dir.cleanup)
pipe_memory = Memory(location=cache_dir.name, verbose=0)


# --- 5. Define Reusable GridSearchCV, Threshold Tuning, & Holdout Evaluation Helper ---
def evaluate_search(search_obj, X_train, y_train, X_valid, y_valid, model_name):
    """Print CV F1 summary, tune decision threshold out-of-fold, and display metrics."""
    # Extract mean CV training and default-threshold validation F1 for the winning parameters
    best_index = search_obj.best_index_
    cv_train_f1 = search_obj.cv_results_["mean_train_score"][best_index]
    default_cv_valid_f1 = search_obj.best_score_

    print(f"Best {model_name} Parameters:", search_obj.best_params_)
    print(f"Best Mean CV Training F1:   {cv_train_f1:.3f}")
    print(f"Best Mean CV Validation F1: {default_cv_valid_f1:.3f}")

    # Tune decision threshold strictly via out-of-fold cross-validation on training data
    best_pipe = search_obj.best_estimator_
    tuned_model = TunedThresholdClassifierCV(
        estimator=best_pipe,
        scoring=f1_purchase,
        cv=cv,
        refit=True,
        n_jobs=-1,
    )
    tuned_model.fit(X_train, y_train)

    threshold_label = (
        "Optimal Probability Threshold"
        if hasattr(best_pipe, "predict_proba")
        else "Optimal Decision Threshold"
    )
    print(f"{threshold_label}: {tuned_model.best_threshold_:.3f}")
    print(f"Tuned Mean CV Validation F1:   {tuned_model.best_score_:.3f}")

    # Preserve best_estimator_ for .named_steps inspection and attach tuned_estimator_
    search_obj.tuned_estimator_ = tuned_model
    cv_valid_f1 = tuned_model.best_score_

    # Generate threshold-tuned class predictions on full training and unseen holdout sets
    y_train_pred = tuned_model.predict(X_train)
    y_valid_pred = tuned_model.predict(X_valid)
    holdout_valid_f1 = f1_score(y_valid, y_valid_pred, zero_division=0)

    # Render side-by-side Accuracy, Precision, Recall, and F1 comparison table
    metrics_comparison = pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
        "Training": [
            accuracy_score(y_train, y_train_pred),
            precision_score(y_train, y_train_pred, zero_division=0),
            recall_score(y_train, y_train_pred, zero_division=0),
            f1_score(y_train, y_train_pred, zero_division=0),
        ],
        "Validation": [
            accuracy_score(y_valid, y_valid_pred),
            precision_score(y_valid, y_valid_pred, zero_division=0),
            recall_score(y_valid, y_valid_pred, zero_division=0),
            holdout_valid_f1,
        ],
    })
    display(metrics_comparison.round(3))

    return cv_train_f1, cv_valid_f1, holdout_valid_f1

### Logistic Regression

In [5]:
# --- 1. Initialize Dedicated Logistic Regression Pipeline ---
# Clone the global preprocessor so fitting GridSearchCV never mutates shared transformers
logistic_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=2000, random_state=42)),
], memory=pipe_memory)

# --- 2. Define Refined Regularization & Class-Weight Grid ---
# Focus around Iteration 1 winner (C=0.1, l1, saga, class_weight={False: 1, True: 2})
param_grid = {
    "model__solver": ["liblinear", "saga"],
    "model__penalty": ["l1", "l2"],
    "model__C": [0.03, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3],
    "model__class_weight": [
        {False: 1, True: 1.5},
        {False: 1, True: 1.8},
        {False: 1, True: 2.0},
        {False: 1, True: 2.2},
        {False: 1, True: 2.5},
    ],
}

# --- 3. Run Stratified 5-Fold Grid Search Exclusively on Training Data ---
logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
logistic_search.fit(X_train, y_train)

# --- 4. Evaluate Cross-Validation & Holdout Performance ---
_, _, lr_valid_f1 = evaluate_search(
    logistic_search, X_train, y_train, X_valid, y_valid, "Logistic Regression"
)

# --- 5. Inspect Top Learned Feature Weights by Magnitude ---
best_logistic_pipe = logistic_search.best_estimator_
feature_names = best_logistic_pipe.named_steps["prep"].get_feature_names_out()
weights = best_logistic_pipe.named_steps["model"].coef_[0]

logistic_weights = (
    pd.DataFrame({
        "Feature": feature_names,
        "Weight": weights,
    })
    .assign(Abs_Weight=lambda df: df["Weight"].abs())
    .sort_values(by="Abs_Weight", ascending=False)
    .drop(columns=["Abs_Weight"])
    .reset_index(drop=True)
)

display(logistic_weights.head(15).round(3))

Best Logistic Regression Parameters: {'model__C': 0.08, 'model__class_weight': {False: 1, True: 2.0}, 'model__penalty': 'l1', 'model__solver': 'saga'}
Best Mean CV Training F1:   0.694
Best Mean CV Validation F1: 0.693
Optimal Probability Threshold: 0.501
Tuned Mean CV Validation F1:   0.693


,Metric,Training,Validation
0,Accuracy,0.893,0.871
1,Precision,0.623,0.562
2,Recall,0.780,0.759
3,F1 Score,0.693,0.646


,Feature,Weight
0,num__PageValues,1.561
1,cat__Month_Nov,1.020
2,cat__Month_May,-0.435
3,cat__TrafficType_1,-0.422
4,cat__VisitorType_Returning_Visitor,-0.392
5,num__ExitRates,-0.283
6,cat__TrafficType_3,-0.213
7,cat__Month_Mar,-0.207
8,num__BounceRates,-0.199
9,cat__Month_Sep,0.190


### Naïve Bayes

In [6]:
# --- 1. Initialize Dedicated Naïve Bayes Pipeline & Discretizer ---
Bayes_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("clf", GaussianNB()),
])

discretized_num_pipeline = Pipeline([
    ("bins", KBinsDiscretizer(encode="onehot-dense", subsample=None)),
])

# --- 2. Define Multi-Regime Hyperparameter Grid ---
# Regime A: One-hot discretized bins + BernoulliNB
# Regime B: Yeo-Johnson z-scores + threshold-binarized BernoulliNB
# Regime C: Continuous Yeo-Johnson features + GaussianNB
param_grid = [
    {
        "prep__num": [discretized_num_pipeline],
        "prep__num__bins__n_bins": [3, 5, 8, 10],
        "prep__num__bins__strategy": ["quantile", "uniform"],
        "clf": [BernoulliNB(binarize=None)],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
        "clf__class_prior": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
    {
        "clf": [BernoulliNB()],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
        "clf__binarize": [0.0, 0.25, 0.5, 1.0],
        "clf__class_prior": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
    {
        "clf": [GaussianNB()],
        "clf__var_smoothing": np.logspace(-11, -1, 11),
        "clf__priors": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
]

# --- 3. Run Stratified 5-Fold Grid Search Exclusively on Training Data ---
bayes_search = GridSearchCV(
    estimator=Bayes_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
bayes_search.fit(X_train, y_train)

# --- 4. Evaluate Cross-Validation & Holdout Performance ---
_, _, nb_valid_f1 = evaluate_search(
    bayes_search, X_train, y_train, X_valid, y_valid, "Naïve Bayes"
)

# --- 5. Inspect Learned Class-Conditional Feature Parameters ---
best_bayes_pipe = bayes_search.best_estimator_
feature_names = best_bayes_pipe.named_steps["prep"].get_feature_names_out()
best_clf = best_bayes_pipe.named_steps["clf"]

if isinstance(best_clf, GaussianNB):
    learned_params = pd.DataFrame({
        "Feature": feature_names,
        "Mean (y=0)": best_clf.theta_[0],
        "Mean (y=1)": best_clf.theta_[1],
        "Var (y=0)": best_clf.var_[0],
        "Var (y=1)": best_clf.var_[1],
    })
else:
    feature_prob = np.exp(best_clf.feature_log_prob_)
    learned_params = pd.DataFrame({
        "Feature": feature_names,
        "P(Feature=1 | y=0)": feature_prob[0],
        "P(Feature=1 | y=1)": feature_prob[1],
    })

display(learned_params.head(10).round(3))

Best Naïve Bayes Parameters: {'clf': BernoulliNB(), 'clf__alpha': 1.0, 'clf__binarize': 1.0, 'clf__class_prior': [0.8, 0.2]}
Best Mean CV Training F1:   0.648
Best Mean CV Validation F1: 0.651
Optimal Probability Threshold: 0.491
Tuned Mean CV Validation F1:   0.650


,Metric,Training,Validation
0,Accuracy,0.866,0.847
1,Precision,0.546,0.503
2,Recall,0.795,0.776
3,F1 Score,0.647,0.611


,Feature,P(Feature=1 | y=0),P(Feature=1 | y=1)
0,num__Administrative,0.235,0.383
1,num__Administrative_Duration,0.213,0.343
2,num__Informational,0.190,0.325
3,num__Informational_Duration,0.171,0.298
4,num__ProductRelated,0.140,0.275
5,num__ProductRelated_Duration,0.129,0.255
6,num__BounceRates,0.214,0.029
7,num__ExitRates,0.224,0.024
8,num__PageValues,0.106,0.820
9,num__SpecialDay,0.117,0.039


### K-Nearest Neighbors

In [7]:
# --- 1. Initialize Dedicated K-Nearest Neighbors Pipeline ---
knn_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", KNeighborsClassifier()),
], memory=pipe_memory)

# --- 2. Define Neighborhood Size, Weighting, & Distance Metric Grid ---
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 19, 25, 31],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["manhattan", "euclidean", "cosine"],
}

# --- 3. Run Stratified 5-Fold Grid Search Exclusively on Training Data ---
knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
knn_search.fit(X_train, y_train)

# --- 4. Evaluate Cross-Validation & Holdout Performance ---
_, _, knn_valid_f1 = evaluate_search(
    knn_search, X_train, y_train, X_valid, y_valid, "K-Nearest Neighbors"
)

# --- 5. Inspect Nearest-Neighbor Distances & Labels for Sample Session ---
best_knn_pipe = knn_search.best_estimator_
knn_model = best_knn_pipe.named_steps["model"]

session = X_train.iloc[[0]]
transformed_session = best_knn_pipe.named_steps["prep"].transform(session)
distances, indices = knn_model.kneighbors(transformed_session)

display(pd.DataFrame({
    "Distance": distances[0],
    "Purchased": y_train.iloc[indices[0]].to_numpy(),
}))

Best K-Nearest Neighbors Parameters: {'model__metric': 'euclidean', 'model__n_neighbors': 31, 'model__weights': 'distance'}
Best Mean CV Training F1:   1.000
Best Mean CV Validation F1: 0.665
Optimal Probability Threshold: 0.293
Tuned Mean CV Validation F1:   0.688


,Metric,Training,Validation
0,Accuracy,1.0,0.862
1,Precision,1.0,0.537
2,Recall,1.0,0.759
3,F1 Score,1.0,0.629


,Distance,Purchased
0,0.000000,False
1,1.568383,False
2,1.770997,False
3,1.857021,False
4,1.955126,False
5,2.032179,False
6,2.050457,True
7,2.057456,False
8,2.111908,False
9,2.139702,False


## Support Vector Machine

In [8]:
# --- 1. Initialize Dedicated Support Vector Machine Pipeline ---
# Leave probability=False (default) to avoid 5x Platt-scaling overhead during CV
svm_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", SVC(random_state=42)),
], memory=pipe_memory)

# --- 2. Define Refined RBF Kernel Hyperparameter Grid ---
# Focus around Iteration 1 winner (kernel='rbf', C=2.0, gamma='scale', class_weight={False: 1, True: 2})
param_grid = {
    "model__kernel": ["rbf"],
    "model__C": [0.8, 1.2, 1.5, 2.0, 2.5, 3.0],
    "model__gamma": ["scale", "auto", 0.005, 0.01, 0.02, 0.03],
    "model__class_weight": [
        {False: 1, True: 1.5},
        {False: 1, True: 1.8},
        {False: 1, True: 2.0},
        {False: 1, True: 2.2},
        {False: 1, True: 2.5},
    ],
}

# --- 3. Run Stratified 5-Fold Grid Search Exclusively on Training Data ---
svm_search = GridSearchCV(
    estimator=svm_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
svm_search.fit(X_train, y_train)

# --- 4. Evaluate Cross-Validation & Holdout Performance ---
_, _, svm_valid_f1 = evaluate_search(
    svm_search, X_train, y_train, X_valid, y_valid, "Support Vector Machine"
)

# --- 5. Inspect Support Vector Counts per Class ---
best_svm_pipe = svm_search.best_estimator_
svm_model = best_svm_pipe.named_steps["model"]
print(f"Winning ({svm_model.kernel}) SVM support vectors:", svm_model.n_support_)

Best Support Vector Machine Parameters: {'model__C': 2.0, 'model__class_weight': {False: 1, True: 1.8}, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Best Mean CV Training F1:   0.777
Best Mean CV Validation F1: 0.703
Optimal Decision Threshold: -0.000
Tuned Mean CV Validation F1:   0.704


,Metric,Training,Validation
0,Accuracy,0.926,0.886
1,Precision,0.735,0.617
2,Recall,0.815,0.692
3,F1 Score,0.773,0.652


Winning (rbf) SVM support vectors: [1354  705]


## Decision Tree

In [9]:
# --- 1. Initialize Dedicated Decision Tree Pipeline ---
dt_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", DecisionTreeClassifier(random_state=42)),
], memory=pipe_memory)

# --- 2. Define Impurity, Pre/Post-Pruning, & Class-Weight Grid ---
param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 4, 5, 6, 8, 10, None],
    "model__min_samples_leaf": [5, 10, 15, 25, 50],
    "model__ccp_alpha": [0.0, 0.0005, 0.001, 0.005],
    "model__class_weight": [
        None,
        "balanced",
        {False: 1, True: 2},
        {False: 1, True: 3},
        {False: 1, True: 4},
    ],
}

# --- 3. Run Stratified 5-Fold Grid Search Exclusively on Training Data ---
dt_search = GridSearchCV(
    estimator=dt_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
dt_search.fit(X_train, y_train)

# --- 4. Evaluate Cross-Validation & Holdout Performance ---
_, _, dt_valid_f1 = evaluate_search(
    dt_search, X_train, y_train, X_valid, y_valid, "Decision Tree"
)

# --- 5. Inspect Top 10 Feature Importances ---
best_dt_pipe = dt_search.best_estimator_
dt_model = best_dt_pipe.named_steps["model"]

dt_importance = pd.DataFrame({
    "Feature": best_dt_pipe.named_steps["prep"].get_feature_names_out(),
    "Importance": dt_model.feature_importances_,
})

display(
    dt_importance.sort_values(
        "Importance",
        ascending=False,
    ).head(10).round(3)
)

Best Decision Tree Parameters: {'model__ccp_alpha': 0.001, 'model__class_weight': {False: 1, True: 2}, 'model__criterion': 'entropy', 'model__max_depth': 6, 'model__min_samples_leaf': 50}
Best Mean CV Training F1:   0.716
Best Mean CV Validation F1: 0.695
Optimal Probability Threshold: 0.505
Tuned Mean CV Validation F1:   0.695


,Metric,Training,Validation
0,Accuracy,0.905,0.881
1,Precision,0.669,0.599
2,Recall,0.760,0.696
3,F1 Score,0.711,0.644


,Feature,Importance
8,num__PageValues,0.766
17,cat__Month_Nov,0.066
6,num__BounceRates,0.033
5,num__ProductRelated_Duration,0.027
16,cat__Month_May,0.026
0,num__Administrative,0.021
15,cat__Month_Mar,0.021
7,num__ExitRates,0.015
1,num__Administrative_Duration,0.015
3,num__Informational_Duration,0.003


## Random Forest

In [10]:
# --- 1. Initialize Dedicated Random Forest Pipeline ---
# Leave n_jobs=1 on the estimator so GridSearchCV handles parallelism cleanly
rf_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", RandomForestClassifier(random_state=42)),
], memory=pipe_memory)

# --- 2. Define Streamlined Regularization, Feature Subsampling, & Class-Weight Grid ---
# 54-candidate grid (270 fits) tightening depth and leaf size around Iteration 1 winner
param_grid = {
    "model__n_estimators": [350],
    "model__max_depth": [8, 10, 12],
    "model__min_samples_leaf": [10, 15, 25],
    "model__max_features": [0.4, 0.5],
    "model__class_weight": [
        {False: 1, True: 1.8},
        {False: 1, True: 2.0},
        {False: 1, True: 2.2},
    ],
}

# --- 3. Run Stratified 5-Fold Grid Search Exclusively on Training Data ---
rf_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)

# --- 4. Evaluate Cross-Validation & Holdout Performance ---
_, _, rf_valid_f1 = evaluate_search(
    rf_search, X_train, y_train, X_valid, y_valid, "Random Forest"
)

# --- 5. Inspect Top 10 Feature Importances ---
best_rf_pipe = rf_search.best_estimator_
feature_names = best_rf_pipe.named_steps["prep"].get_feature_names_out()
importances = best_rf_pipe.named_steps["model"].feature_importances_

rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances,
})

display(
    rf_importance.sort_values(
        "Importance",
        ascending=False,
    ).head(10).round(3)
)

Best Random Forest Parameters: {'model__class_weight': {False: 1, True: 2.2}, 'model__max_depth': 12, 'model__max_features': 0.5, 'model__min_samples_leaf': 15, 'model__n_estimators': 350}
Best Mean CV Training F1:   0.754
Best Mean CV Validation F1: 0.710
Optimal Probability Threshold: 0.503
Tuned Mean CV Validation F1:   0.710


,Metric,Training,Validation
0,Accuracy,0.918,0.885
1,Precision,0.706,0.609
2,Recall,0.809,0.720
3,F1 Score,0.754,0.660


,Feature,Importance
8,num__PageValues,0.692
7,num__ExitRates,0.073
5,num__ProductRelated_Duration,0.049
4,num__ProductRelated,0.035
17,cat__Month_Nov,0.031
6,num__BounceRates,0.028
0,num__Administrative,0.021
1,num__Administrative_Duration,0.020
3,num__Informational_Duration,0.006
16,cat__Month_May,0.006


## Gradient Boost

In [11]:
# --- 1. Initialize Dedicated Gradient Boosting Pipeline ---
gb_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", GradientBoostingClassifier(random_state=42)),
], memory=pipe_memory)

# --- 2. Define Refined Shrinkage, Tree Complexity, & Stochastic Subsampling Grid ---
# Expand around Iteration 1 boundary hits (learning_rate=0.03, min_samples_leaf=15, subsample=1.0)
param_grid = {
    "model__n_estimators": [150, 180, 220],
    "model__learning_rate": [0.02, 0.03, 0.04],
    "model__max_depth": [3, 4],
    "model__min_samples_leaf": [15, 25, 35],
    "model__subsample": [0.85, 0.95, 1.0],
    "model__max_features": [0.4, 0.5],
}

# --- 3. Run Stratified 5-Fold Grid Search Exclusively on Training Data ---
gb_search = GridSearchCV(
    estimator=gb_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
gb_search.fit(X_train, y_train)

# --- 4. Evaluate Cross-Validation & Holdout Performance ---
_, _, gb_valid_f1 = evaluate_search(
    gb_search, X_train, y_train, X_valid, y_valid, "Gradient Boost"
)

# --- 5. Inspect Top 10 Feature Importances ---
best_gb_pipe = gb_search.best_estimator_
gb_model = best_gb_pipe.named_steps["model"]

gb_importance = pd.DataFrame({
    "Feature": best_gb_pipe.named_steps["prep"].get_feature_names_out(),
    "Importance": gb_model.feature_importances_,
})

display(
    gb_importance.sort_values(
        "Importance",
        ascending=False,
    ).head(10).round(3)
)

Best Gradient Boost Parameters: {'model__learning_rate': 0.04, 'model__max_depth': 4, 'model__max_features': 0.4, 'model__min_samples_leaf': 15, 'model__n_estimators': 220, 'model__subsample': 1.0}
Best Mean CV Training F1:   0.777
Best Mean CV Validation F1: 0.686
Optimal Probability Threshold: 0.399
Tuned Mean CV Validation F1:   0.705


,Metric,Training,Validation
0,Accuracy,0.929,0.894
1,Precision,0.765,0.648
2,Recall,0.782,0.689
3,F1 Score,0.773,0.668


,Feature,Importance
8,num__PageValues,0.679
7,num__ExitRates,0.059
5,num__ProductRelated_Duration,0.046
6,num__BounceRates,0.041
4,num__ProductRelated,0.034
17,cat__Month_Nov,0.033
0,num__Administrative,0.028
1,num__Administrative_Duration,0.020
3,num__Informational_Duration,0.009
55,cat__VisitorType_Returning_Visitor,0.007


## Model Comparison

In [12]:
# --- 1. Collect All Fitted Search Objects & Holdout F1 Scores ---
searches = [
    ("Logistic Regression", logistic_search, lr_valid_f1),
    ("Naïve Bayes", bayes_search, nb_valid_f1),
    ("K-Nearest Neighbors", knn_search, knn_valid_f1),
    ("Support Vector Machine", svm_search, svm_valid_f1),
    ("Decision Tree", dt_search, dt_valid_f1),
    ("Random Forest", rf_search, rf_valid_f1),
    ("Gradient Boost", gb_search, gb_valid_f1),
]

# --- 2. Assemble Cached Training, Threshold-Tuned CV, and Holdout F1 Summary Table ---
model_results = [
    {
        "Model": name,
        "Mean training F1": search.cv_results_["mean_train_score"][search.best_index_],
        "Mean CV F1": getattr(search, "tuned_estimator_", search).best_score_,
        "Holdout Validation F1": holdout_f1,
    }
    for name, search, holdout_f1 in searches
]

# --- 3. Display Final Model Comparison ---
display(pd.DataFrame(model_results).round(3))

,Model,Mean training F1,Mean CV F1,Holdout Validation F1
0,Logistic Regression,0.694,0.693,0.646
1,Naïve Bayes,0.648,0.650,0.611
2,K-Nearest Neighbors,1.000,0.688,0.629
3,Support Vector Machine,0.777,0.704,0.652
4,Decision Tree,0.716,0.695,0.644
5,Random Forest,0.754,0.710,0.660
6,Gradient Boost,0.777,0.705,0.668
